# SourceStretching — the M1 induction source term $S=(b\cdot\nabla)u$

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``source_stretching``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

The `sources { induction_stretching { } }` block adds $S=(b\cdot\nabla)u$ to
the right-hand side. The module transports a single scalar component, so with
a uniform $b=B_0$ and a **divergence-free** velocity $u=(x,-y)$ (so
$\nabla u=\mathrm{diag}(1,-1)$), convection and diffusion vanish identically
and

$$\frac{d b}{dt}=B_0\frac{\partial u_0}{\partial x}=B_0,$$

i.e. $b(t)=B_0e^{t}$. A wrong factor, a transposed gradient index, or a
dropped term changes the exponential rate, so this is a sharp absolute check
of the source magnitude and sign.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| — | M1 stretching source $S=(b\cdot\nabla)u$ | $b\sim e^{t}$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/source_stretching.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = 17                        # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 17, "ny": 17},            # the baseline mesh
    "standard": {"nx": 2*NX0-1, "ny": 2*NY0-1},
    "fine":     {"nx": 4*NX0-3, "ny": 4*NY0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("source_stretching.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition source_stretching.data 2 -> PAR_source_stretching.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
t, b0 = read_son(case.dataFileName_, "SONDE_B0")
b0_0 = b0[0]
expected = b0_0 * np.exp(t)
err = np.max(np.abs(b0 - expected)) / abs(b0_0)
print(f"b0(0)={b0_0:.4f}  b0(t_end)={b0[-1]:.4f}  expected exp(t_end)={expected[-1]:.4f}")
print(f"max |b - B0 exp(t)| / B0 : {err:.3e}")
# the source must reproduce exp(t) to a few percent (first-order in dt
# for an explicit term; dt=2e-3 keeps the error well below 1%)
assert err < 0.01, f"induction source rate wrong: {err:.3e}"
print("OK: M1 source S=(b.grad)u has the correct magnitude and sign.")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(t, b0 / b0_0, label="b/B0 (simulated)")
plt.plot(t, np.exp(t), "k--", label="exp(t) (analytic)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()


## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`17×17`),
`standard` = cells ×2 (`33×33`), `fine` = cells ×4 (`65×65`).

The growth $b(t) = B_0 e^{t}$ saturates at the **explicit-Euler temporal**
error of the lagged source (first order in $\Delta t$, common to every
mesh). To cancel it, the scan compares $b/b_0$ against the exact
explicit-Euler growth on the probe's own time grid,
$\mathrm{ref}(t_n) = \prod_j (1 + \Delta t_j)$. The residual is the
**spatial** error of the $(b\cdot\nabla)u$ reconstruction — which is exact
for the linear data velocity $u=(x,-y)$ at interior cells (Green–Gauss),
so the cancelled residual has **no spatial contribution**: its floor is the
level-independent time-scheme convergence slack ($\sim10^{-3}$ here). The
scan asserts a fixed cancelled bound at **every** level — a spatial error
would appear as growth under refinement, which the bound rules out.

In [ ]:
SONDE_TAG_SCAN = "SONDE_B0"

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def explicit_ref(t):
    """Exact explicit-Euler growth ratio on the probe time grid (lam = 1)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 + (t[i] - t[i-1])
    return ref

def cancelled_error(subdir, case_name):
    """max |b(t)/b(0) - explicit-Euler growth| over all recorded times."""
    t, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    if b[0] == 0:
        return 0.0
    return float(np.max(np.abs(b / b[0] - explicit_ref(t))))

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (17, 17), "standard": (33, 33), "fine": (65, 65)}
scan_err = {lvl: cancelled_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes     |ratio - explicit-Euler ref|")
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny:<4} {scan_err[lvl]:.3e}")
    # the residual floor is the implicit-iteration convergence slack
    # (level-independent); a spatial error would appear as growth
    # under refinement, asserted absent via the fixed bound.
    assert scan_err[lvl] < 5e-3, \
        f"cancelled stretching bound violated at {lvl}: {scan_err[lvl]:.3e}"
print("OK: cancelled bound holds at every level (grad(u) exact for the linear data velocity).")

## Computer Performance


In [ ]:
run.tablePerf()